# Regressão Linear - Preço de Imóveis (UCI Real Estate Valuation)
Mesmo tipo de análise do app das corridas de Uber (regressão linear, R², RMSE, MAE, resíduos).

Dataset: https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set
Baixe `Real estate valuation data set.xlsx` e coloque na mesma pasta deste notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_excel("Real estate valuation data set.xlsx")

# Renomeia colunas para nomes mais simples
df = df.rename(columns={
    "X1 transaction date": "data_transacao",
    "X2 house age": "idade_imovel",
    "X3 distance to the nearest MRT station": "distancia_estacao",
    "X4 number of convenience stores": "num_lojas",
    "X5 latitude": "latitude",
    "X6 longitude": "longitude",
    "Y house price of unit area": "preco_unidade"
})
df.head()

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

In [ ]:
# Distribuição do preço e relação com a distância da estação
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df["preco_unidade"], bins=30, color="#1DB954", ax=axes[0])
axes[0].set_title("Distribuição do Preço por Unidade de Área")

sns.scatterplot(x="distancia_estacao", y="preco_unidade", data=df, alpha=0.6, ax=axes[1])
axes[1].set_title("Preço x Distância da Estação de Metrô")
plt.tight_layout()
plt.show()

In [ ]:
# Correlação entre as variáveis
plt.figure(figsize=(6,5))
sns.heatmap(df.drop(columns=["No"], errors="ignore").corr(), annot=True, fmt=".2f", cmap="RdBu_r")
plt.title("Mapa de Correlação")
plt.show()

In [ ]:
# Seleciona variáveis preditoras e alvo
features = ["idade_imovel", "distancia_estacao", "num_lojas", "latitude", "longitude"]
X = df[features].values
y = df["preco_unidade"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Treina a regressão linear
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

In [ ]:
# Coeficientes do modelo
coef_df = pd.DataFrame({"Variavel": features, "Coeficiente": model.coef_}).sort_values("Coeficiente")
sns.barplot(x="Coeficiente", y="Variavel", data=coef_df, palette="coolwarm")
plt.title("Coeficientes do Modelo")
plt.show()

In [ ]:
# Real x Predito e resíduos
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(y_test, y_pred, alpha=0.6, color="#1DB954")
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "r--")
axes[0].set_xlabel("Preço Real")
axes[0].set_ylabel("Preço Predito")
axes[0].set_title("Real x Predito")

sns.histplot(residuos, bins=30, color="#FF6B35", ax=axes[1])
axes[1].set_title("Distribuição dos Resíduos")
plt.tight_layout()
plt.show()

In [ ]:
# Equação do modelo e exemplo de predição
eq = f"preco = {model.intercept_:.4f}"
for feat, coef in zip(features, model.coef_):
    eq += f" + ({coef:.4f} * {feat})"
print(eq)

# Exemplo: imóvel com 10 anos, 500m da estação, 5 lojas por perto
novo_imovel = np.array([[10, 500, 5, 24.98, 121.54]])
pred = model.predict(novo_imovel)[0]
print(f"Preço estimado por unidade de área: {pred:.2f}")